In [1]:
import numpy as np
from pyscf import gto, scf, mp, cc

lno_num = 4
lno_list = [3e-4,1e-4,3e-5,1e-5]
lno_thresh = lno_list[lno_num-1]

####  test H2 monomers ####
a = 2 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 3 # set as integer multiple of monomers
spin = 2 # spin per monomer
elmt = 'O'
unit = 'B'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=40000,
            )

mf = scf.UHF(mol).density_fit()
mf.kernel()

stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'UHF Energy: {mf.e_tot}, stability {stable}')
        break

mymp = mp.MP2(mf).set_frozen()
mymp.kernel()

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()

print(f"HF   : {mf.e_tot}")
print(f"MP2  : {mymp.e_tot}")
print(f"CCSD : {mycc.e_tot}")

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Thu Jun 25 14:34:42 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 48
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 6
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      uni

In [2]:
print(f"HF per M   : {mf.e_tot/nc}")
print(f"MP2 per M  : {mymp.e_tot/nc}")
print(f"CCSD per M : {mycc.e_tot/nc}")

HF per M   : -148.964269257585
MP2 per M  : -149.02794902501543
CCSD per M : -149.03982586224402


In [6]:
import jax
jax.config.update("jax_enable_x64", True)
import opt_einsum as oe

In [136]:
from afqmc.lno_afqmc import lno_afqmc, tools
from pyscf.data import elements
lo_coeff, frag_lolist, atm_center = tools.iao_localization(mf)

In [ ]:
options = {
           'n_blocks': 10,
           'n_walkers': 300,
           'max_memory': 2000,
           'mix_precision': False,
           'n_batch': 1,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'upt2ccsd',
           }

In [130]:
import numpy as np
from jax import random

from pyscf import lib
from pyscf.lno import lnoccsd
from pyscf.lno import ulnoccsd
from collections.abc import Iterable

# from afqmc import config
from afqmc.lno_afqmc import lno_afqmc
from afqmc.lno_afqmc import prep, tools
from afqmc.lno_afqmc import mod_lnoccsd

from functools import partial
import time, gc, pickle, os

print = partial(print, flush=True)

# mf
# lo_coeff = None
# frag_lolist = None
nfrozen = elements.chemcore(mf.mol)
thresh = 1e-8
qmc_options = options
chol_cut = 1e-12
target_sto_error = 0
run_frag_list = [0] 
atom_group = atm_center
plot_las = False

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

if spin_type == "unrestricted":
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mf = mlno._scf
else:
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
mlno.lno_thresh = [thresh*10, thresh]
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag_tot = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag_tot)

frag_lolist = [frag_lolist[i] for i in run_frag_list]
nfrag_run = len(frag_lolist)

lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag_tot

seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                        shape=(nfrag_tot,), 
                        minval=0, 
                        maxval=100*nfrag_tot
                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag_run
las_size = [None]*nfrag_run
lno_emp2 = np.zeros(nfrag_run, dtype='float64')
lno_ecc  = np.zeros(nfrag_run, dtype='float64')
lno_eqmc = np.zeros(nfrag_run, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')
ccsd_time = np.zeros(nfrag_run, dtype='float64')
qmc_time = np.zeros(nfrag_run, dtype='float64')

mol = mf.mol

# Loop over fragment
for ifrag, loidx in enumerate(frag_lolist):
    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {run_frag_list[ifrag]+1}/({nfrag_run},{nfrag_tot}) "
    print(msg.center(width, '='))
    if atom_group is not None:
        loc_ctr = f"{atom_group[run_frag_list[ifrag]]}"
        print(f"Center Atom {loc_ctr}")
    else:
        loc_ctr = None
    
    print(f"PySCF NumPy Threads = {lib.num_threads()}")

    if spin_type == "unrestricted":
        orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
        lno_param = [
            [
                {
                    'thresh': (
                        lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                        else lno_thresh[i]
                    ),
                    'pct_occ': (
                        lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                        else lno_pct_occ[i]
                    ),
                    'norb': (
                        lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                        else lno_norb[ifrag][i]
                    ),
                } for i in [0, 1]
            ] for s in range(2)
        ]
    else:
        orbloc = lo_coeff[:,loidx]
        lno_param = [{
            'thresh': lno_thresh[i],
            'pct_occ': lno_pct_occ[i],
            'norb': lno_norb[ifrag][i]
            } for i in [0,1]]

    # M = <orbloc|canactocc> (M^dagger M)u = eu
    # u|canactocc> => orbtial in/out the space spanned by |orbloc>
    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

    mo_occ = mlno.mo_occ

    if spin_type == "unrestricted":
        if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
            lno_elec_type = 'alpha'
        elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
            lno_elec_type = 'beta'
        else:
            lno_elec_type = 'mixed'
        print(f'LNO-Frgament Spin Type = {lno_elec_type}')

        if loc_ctr is None:
            ao_max_a = prep.ao_comp(mf, orbloc[0])
            ao_max_b = prep.ao_comp(mf, orbloc[1])
            loc_ctr = ao_max_a + ao_max_b
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
        occidxa = mo_occ[0] > 1e-10
        occidxb = mo_occ[1] > 1e-10
        moidxa, moidxb = maskact
        nactocc_a = int(np.sum(moidxa & occidxa))
        nactvir_a = int(np.sum(moidxa & ~occidxa))
        nactocc_b = int(np.sum(moidxb & occidxb))
        nactvir_b = int(np.sum(moidxb & ~occidxb))
        nactocc = [nactocc_a, nactocc_b]
        nactvir = [nactvir_a, nactvir_b]
        lno_active_a = np.array([i for i in range(mol.nao) if i not in lno_frozen[0]])
        lno_active_b = np.array([i for i in range(mol.nao) if i not in lno_frozen[1]])
        lno_active = [lno_active_a, lno_active_b]
        lno_tot = [len(lno_active_a), len(lno_active_b)]
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    else:
        print(f'LNO-Frgament Spin Type = restricted')
        if loc_ctr is None:
            loc_ctr = prep.ao_comp(mf, orbloc)
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
        lno_active = np.array([i for i in range(mol.nao) if i not in lno_frozen])
        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        lno_tot = len(lno_active)
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    
    if plot_las:
        tools.plot_density(mol, orbloc, lno_coeff, lno_active, spin_type, idx = run_frag_list[ifrag]+1)

    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 =\
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
    time1 = time.perf_counter()
    lnocc_time = time1 - time0

    print(f'LNO-MP2 Orbital Energy:  {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')
    print(f"LNO-CCSD time:           {lnocc_time:.2f} s")

    las_center[ifrag] = loc_ctr
    las_size[ifrag] = lno_tot
    lno_emp2[ifrag] = eorb_mp2
    lno_ecc[ifrag] = eorb_cc
    ccsd_time[ifrag] = lnocc_time

    # project onto center lo space
    # <lno_actocc|orbloc> <orbloc|lno_actocc>
    if spin_type == "unrestricted":
        prjlo = [uocc_loc[0] @ uocc_loc[0].T.conj(),
                    uocc_loc[1] @ uocc_loc[1].T.conj()]
        qmc_options["trial"] = trial_base
        if 'ad' not in trial_base:
            if lno_elec_type == 'alpha':
                qmc_options["trial"] += '_alpha'
            elif lno_elec_type == 'beta':
                qmc_options["trial"] += '_beta'
    else:
        prjlo = uocc_loc @ uocc_loc.T.conj()

    qmc_options["seed"] = seeds[ifrag]
    prep.prep_afqmc_integral(
        mf,
        lno_coeff,
        t1,
        t2,
        lno_frozen,
        prjlo,
        qmc_options,
        chol_cut=chol_cut
        )
    
    lno_afqmc.run_lnoafqmc(qmc_options)
    outfile = f'fragment.out{run_frag_list[ifrag]+1}'
    os.system(f'mv afqmc.out {outfile}')
    with open(outfile, "r") as f:
        for line in f:
            if "Blocked AFQMC/pt2CCSD Orbital Energy" in line:
                eorb_afqmc = float(line.split()[-3])
                eorb_afqmc_err = float(line.split()[-1])
            if "total run time" in line:
                lnoqmc_time = float(line.split()[-1])
    lno_eqmc[ifrag] = eorb_afqmc
    lno_eqmc_err[ifrag] = eorb_afqmc_err
    qmc_time[ifrag] = lnoqmc_time

    header = f' Fragment{run_frag_list[ifrag]+1} Results '
    width = 80  # pick a consistent total width
    with open(outfile, 'a') as f:
        f.write('\n')
        f.write(f'{header:=^{width}}\n')
        f.write("\t LNO Center " + loc_ctr + "\n")
        f.write('-' * width + '\n')
        f.write(f'\t LNO-Active Space electrons: {nactocc} | orbitals: {nactocc+nactvir} \n')
        f.write(f'\t LNO-MP2 Orbital Energy:   {eorb_mp2:.8f} \n')
        f.write(f'\t LNO-CCSD Orbital Energy:  {eorb_cc:.8f} \n')
        f.write(f'\t LNO-AFQMC Orbital Energy: {eorb_afqmc:.6f} +/- {eorb_afqmc_err:.6f} \n')
        f.write(f'\t LNO-CCSD Time:  {lnocc_time:.2f} \n')
        f.write(f'\t LNO-AFQMC Time: {lnoqmc_time:.2f} \n')
        f.write('=' * width + '\n')
    jax.clear_caches()
    gc.collect()

las_size = np.array(las_size, dtype=np.int32)
las_max = las_size.max()
# convert to list of string for print
las_size = list(map(lambda row: f"{row}", las_size))
e_mp2 = np.sum(lno_emp2)
e_ccsd = np.sum(lno_ecc)
e_afqmc = np.sum(lno_eqmc)
e_afqmc_err = np.sqrt(np.sum(lno_eqmc_err**2))
tot_ccsd_time = np.sum(ccsd_time)
tot_qmc_time = np.sum(qmc_time)

with open(f'lno_result.out', 'w') as f:
    width = 100
    f.write('=' * width + '\n')
    f.write(f'{"LNO-AFQMC Results":^{width}}\n')
    f.write('=' * width + '\n')

    f.write(f'{"Frag":>4s}  {"LAS Center":>14s}  {"LAS_SIZE":>8s}  '
            f'{"E(MP2)":>10s}  {"E(CCSD)":>10s}  '
            f'{"E(AFQMC)":>10s}  {"Error":>8s}  '
            f'{"t(CCSD)":>8s}  {"t(AFQMC)":>8s}\n')
    f.write('-' * width + '\n')
    
    for n, i in enumerate(run_frag_list):
        f.write(f"{i+1:4d}  {las_center[n]:>14s}  {las_size[n]:8s}  "
                f"{lno_emp2[n]:10.8f}  {lno_ecc[n]:10.8f}  "
                f"{lno_eqmc[n]:10.6f}  {lno_eqmc_err[n]:8.6f}  "
                f"{ccsd_time[n]:8.2f}  {qmc_time[n]:8.2f}\n")
    
    f.write('-' * width + '\n')
    f.write(f'{"Summarize Fragments":^{width}}\n')
    f.write('-' * width + '\n')
    lno_thresh_str = "[" + ", ".join(f"{x:.2e}" for x in lno_thresh) + "]"
    f.write(f'{"LNO-Thresh":<20} {"Max LAS":>8} '
            f'{"E[MP2]":>12} {"E[CCSD]":>12} '
            f'{"E[AFQMC]":>10} {"Err[AFQMC]":>10} '
            f'{"CCSD-Time":>10} {"AFQMC-Time":>10}\n')

    f.write(f'{lno_thresh_str:<20} {las_max:>8} '
            f'{e_mp2:>12.8f} {e_ccsd:>12.8f} '
            f'{e_afqmc:>10.6f} {e_afqmc_err:>10.6f} '
            f'{tot_ccsd_time:>10.2f} {tot_qmc_time:>10.2f}\n')
    
    f.write('=' * width + '\n\n')



====================== unrestricted LNO-FRAGMENT 1/(1,6) =======================
Center Atom O
PySCF NumPy Threads = 16
LNO-Frgament Spin Type = mixed
LAS occupied orbitals:  [7, 5]
LAS virtual orbitals:   [1, 3]
LAS total size:         [8, 8]
LNO-MP2 Orbital Energy:  -0.03183988
LNO-CCSD Orbital Energy: -0.03777830
LNO-CCSD time:           0.06 s
Calculating Effective Active Space One-electron Integrals
Building JK matrix
build JK time: 0.469052 s
build ecore time: 0.078774 s
build h1eff time: 0.583741 s
Generating Cholesky Integrals
Constracting cLAS that span both Alpha and Beta active space
Naive cLAS Shape:  (30, 16)
Orthonormalize cLAS shape: (30, 16)
Smallest cLAS SVD Singular values: 2.372482198379609e-16
cLAS projection torr: 1e-08
Minimum size of cLAS to span both Alpha and Beta LAS: 10
cLAS projection loss: (4.55e-13, 4.66e-13)
True Common LAS Shape:  (30, 10)
Composing AO ERIs from DF basis
Raw CDERI in cLAS shape: (516, 55)
Cholesky cutoff is: 1e-12
Compress CDERI into C

In [131]:
ham_data, prop, trial, wave_data, sampler, options = (prep.prep_afqmc_run())
wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = trial._build_measurement_intermediates(ham_data, wave_data)
ham_data = prop._build_propagation_intermediates(ham_data, trial, wave_data)
prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers = None)



QMC System
Number of electrons:        (7, 5)
Number of orbitals:         (8, 8)
Number of Cholesky Vectors: 53
Maximum memory per walker:            6.67 MB
Maximum number of Cholesky per chunk: 3413
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         53
Number of padding Cholesky:           0

QMC Parameters
n_prop_steps    -              50
n_blocks        -              10
n_walkers       -             300
max_memory      -            2000
mix_precision   -           False
n_batch         -               1
seed            -             442
walker_type     -             uhf
trial           -        upt2ccsd
max_error       -               0
dt              -           0.005
eql_time        -              20
nchol_chunk     -              53
n_exp_terms     -               6


In [132]:
t1a, t1b = mycc.t1
t2aa, t2ab, t2bb = mycc.t2
e0t1 = mycc.energy((t1a, t1b),(t2aa*0, t2ab*0, t2bb*0))
print(e0t1, e0t1/nc/2)
print(ham_data["e0t1orb"])

1.546921155237998e-05 2.5782019253966637e-06
2.5781747173715157e-06


In [133]:
mol = mf.mol
nocc_a = int(sum(mf.mo_occ[0]))
actfrag_a = np.array([i for i in range(mol.nao) if i not in lno_frozen[0]])
frzocc_a = np.array([i for i in range(nocc_a) if i in lno_frozen[0]])
actocc_a = np.array([i for i in range(nocc_a) if i in actfrag_a])
actvir_a = np.array([i for i in range(nocc_a,mol.nao) if i in actfrag_a])
nfrzocc_a = len(frzocc_a)
nactocc_a = len(actocc_a)
nactvir_a = len(actvir_a)
nactorb_a = len(actfrag_a)
nocc_b = int(sum(mf.mo_occ[1]))
actfrag_b = np.array([i for i in range(mol.nao) if i not in lno_frozen[1]])
frzocc_b = np.array([i for i in range(nocc_b) if i in lno_frozen[1]])
actocc_b = np.array([i for i in range(nocc_b) if i in actfrag_b])
actvir_b = np.array([i for i in range(nocc_b,mol.nao) if i in actfrag_b])
nfrzocc_b = len(frzocc_b)
nactocc_b = len(actocc_b)
nactvir_b = len(actvir_b)
nactorb_b = len(actfrag_b)
lno_occ_act = (lno_coeff[0][:,actocc_a], lno_coeff[1][:,actocc_b])

In [134]:
lno_occ_act[1].shape

(30, 5)

In [137]:
frag_lolist[1]

[[5, 6, 7, 8, 9], [5, 6, 7, 8, 9]]

In [138]:
frag1 = (lo_coeff[0][:,frag_lolist[0][0]], lo_coeff[1][:,frag_lolist[0][1]])
frag2 = (lo_coeff[0][:,frag_lolist[1][0]], lo_coeff[1][:,frag_lolist[1][1]])

In [139]:
s1e = mf.get_ovlp()
frag1_lno = (frag1[0].T @ s1e @ lno_occ_act[0], frag1[1].T @ s1e @ lno_occ_act[1])
pfrag1 = (frag1_lno[0].T @ frag1_lno[0], frag1_lno[1].T @ frag1_lno[1])
frag2_lno = (frag2[0].T @ s1e @ lno_occ_act[0], frag2[1].T @ s1e @ lno_occ_act[1])
pfrag2 = (frag2_lno[0].T @ frag2_lno[0], frag2_lno[1].T @ frag2_lno[1])

In [140]:
pfrag1[1] - prjlo[1]

array([[-1.11022302e-16,  2.49800181e-16, -5.55111512e-17,
         1.97215226e-31, -1.67632942e-30],
       [ 2.49800181e-16, -2.22044605e-16,  1.11022302e-16,
        -3.94430453e-31,  7.88860905e-31],
       [-5.55111512e-17,  1.11022302e-16, -1.11022302e-16,
        -3.94430453e-31, -8.87468518e-31],
       [ 1.97215226e-31, -3.94430453e-31, -3.94430453e-31,
         0.00000000e+00,  2.77555756e-17],
       [-1.67632942e-30,  7.88860905e-31, -8.87468518e-31,
         2.77555756e-17, -1.66533454e-16]])

In [141]:
from afqmc import slater_tools
from jax import jit, jvp, lax
from jax import numpy as jnp

@jit
def u_energy_corr_fock(bra, ket, e0, fock, chol):
    '''
    correlation energy E_corr = <bra|H-E0|ket>/<bra|ket> 
    '''
    chola, cholb = chol
    if len(chola.shape) == 3:
        chola = chola.reshape(1,*chola.shape)
    if len(cholb.shape) == 3:
        cholb = cholb.reshape(1,*cholb.shape)

    norba, nocca = ket[0].shape
    norbb, noccb = ket[1].shape
    rot_focka = fock[0][:nocca,nocca:]
    rot_fockb = fock[1][:noccb,noccb:]
    rot_chola = chola[:,:,:nocca,nocca:] # shape(nchol,nocc,nvir)
    rot_cholb = cholb[:,:,:noccb,noccb:]

    greena = (ket[0].dot(jnp.linalg.inv(ket[0][:nocca,:]))).T
    greenb = (ket[1].dot(jnp.linalg.inv(ket[1][:noccb,:]))).T
    greena = greena[:nocca, nocca:]
    greenb = greenb[:noccb, noccb:]

    e1a = oe.contract('ia,ia->', greena, rot_focka, backend="jax")
    e1b = oe.contract('ia,ia->', greenb, rot_fockb, backend="jax")
    e1 = e1a + e1b # should be zero for mean-field solution bra

    def scan_chol(carry, x):
        rot_chola_c, rot_cholb_c = x
        lga_c = oe.contract('gia,ja->gij', rot_chola_c, greena, backend="jax")
        lgb_c = oe.contract('gia,ja->gij', rot_cholb_c, greenb, backend="jax")
        tr_lga_c = oe.contract('gii->g',lga_c, backend="jax")
        tr_lgb_c = oe.contract('gii->g',lgb_c, backend="jax")
        tr_lg_c = tr_lga_c + tr_lgb_c
        e_col_c = jnp.sum(tr_lg_c**2) / 2
        e_exc_c = (oe.contract('gij,gji->',lga_c,lga_c, backend="jax")
                    + oe.contract('gij,gji->',lgb_c,lgb_c, backend="jax")) / 2
        ecorr_c = e_col_c - e_exc_c
        carry += ecorr_c
        return carry, 0.0
    
    e2, _ = lax.scan(scan_chol, 0.0, (rot_chola, rot_cholb))

    return e0 + e1 + e2, e1, e2

@jit
def u_energy_corr_frag(bra, ket, fock, chol, pfrag):
    '''
    calculate the correlation energy of the Hamiltonian
    '''

    chola, cholb = chol
    if len(chola.shape) == 3:
        chola = chola.reshape(1,*chola.shape)
    if len(cholb.shape) == 3:
        cholb = cholb.reshape(1,*cholb.shape)

    norba, nocca = ket[0].shape 
    norbb, noccb = ket[1].shape
    pfraga, pfragb = pfrag
    rot_focka = fock[0][:nocca,nocca:]
    rot_fockb = fock[1][:noccb,noccb:]
    rot_chola = chola[:,:,:nocca,nocca:] # shape(nchunk,nchol_chunk,nocc,nvir)
    rot_cholb = cholb[:,:,:noccb,noccb:]

    gfa = (ket[0].dot(jnp.linalg.inv(ket[0][:nocca,:]))).T
    gfb = (ket[1].dot(jnp.linalg.inv(ket[1][:noccb,:]))).T
    gfa = gfa[:nocca, nocca:]
    gfb = gfb[:noccb, noccb:]
    e1a = oe.contract('ia,ik,ka->', gfa, pfraga, rot_focka, backend="jax")
    e1b = oe.contract('ia,ik,ka->', gfb, pfragb, rot_fockb, backend="jax")
    e1 = e1a + e1b

    def scan_chunk(carry, x):
        rot_chola_c, rot_cholb_c = x
        # explicit contraction within the chunk (g is chunk-local aux index)
        lga = oe.contract('gia,ja->gij', rot_chola_c, gfa, backend="jax")
        lgb = oe.contract('gia,ja->gij', rot_cholb_c, gfb, backend="jax")
        tr_lga = oe.contract('gii->g', lga, backend="jax")
        tr_lgb = oe.contract('gii->g', lgb, backend="jax")
        lga_frag = oe.contract('gik,ik->g', lga, pfraga, backend="jax")
        lgb_frag = oe.contract('gik,ik->g', lgb, pfragb, backend="jax")
        e2aa = oe.contract('g,g->', lga_frag, tr_lga, backend="jax") \
            - oe.contract('gij,gjk,ik->', lga, lga, pfraga, backend="jax")
        e2ab = oe.contract('g,g->', lga_frag, tr_lgb, backend="jax")
        e2ba = oe.contract('g,g->', lgb_frag, tr_lga, backend="jax")
        e2bb = oe.contract('g,g->', lgb_frag, tr_lgb, backend="jax") \
            - oe.contract('gij,gjk,ik->', lgb, lgb, pfragb, backend="jax")
        carry += 0.5 * (e2aa + e2ab + e2ba + e2bb)
        return carry, 0.0

    e2, _ = lax.scan(scan_chunk, 0.0, (rot_chola, rot_cholb))

    return e1 + e2, e1, e2


In [142]:
walkers_up, walkers_dn = prop_data["walkers"]
init_walker = (prop_data["walkers"][0][0], prop_data["walkers"][1][0])

In [143]:
from afqmc import integral
bra = wave_data["mo_coeff"]
h0 = ham_data["h0"]
h1 = ham_data["h1"]
norba = h1[0].shape[0]
norbb = h1[1].shape[1]
(chola, cholb) = ham_data["chol"]
chola = chola.reshape(-1,norba,norba)
cholb = cholb.reshape(-1,norbb,norbb)
chol = (chola, cholb)
nocc = (init_walker[0].shape[1], init_walker[1].shape[1])
fock = integral.get_ufock(nocc, h1, chol)

In [144]:
ket_up = jnp.array(np.random.rand(*init_walker[0].shape)) + 1j*jnp.array(np.random.rand(*init_walker[0].shape))
ket_dn = jnp.array(np.random.rand(*init_walker[1].shape)) + 1j*jnp.array(np.random.rand(*init_walker[1].shape))
ket = (ket_up, ket_dn)
e01 = slater_tools.u_energy(bra, ket, h0, h1, chol)
e02, e12, e22 = u_energy_corr_fock(bra, ket, mf.e_tot, fock, chol)
print(e01-mf.e_tot)
print(e02-mf.e_tot,e12, e22)
nocca, noccb = nocc
id = (jnp.eye(nocca), jnp.eye(noccb)) 
e03, e13, e23 = u_energy_corr_frag(bra, ket, fock, chol, pfrag1)
print(e03, e13, e23)
e04, e14, e24 = u_energy_corr_frag(bra, ket, fock, chol, pfrag2)
print(e04, e14, e24)
print(e03+e04, e13+e14, e23+e24)

(0.356622764944575-0.5275353392143296j)
(0.35662276494304024-0.527535339214334j) (-1.7840848650782678e-10-1.0733013294820222e-07j) (0.3566227651214735-0.527535231884201j)
(0.04860822079294501+0.1006487696789429j) (-8.679200735580799e-09-7.48469600206322e-09j) (0.04860822947214574+0.1006487771636389j)
(0.3080145441501207-0.6281841088932782j) (8.500792249073146e-09-9.984543694613884e-08j) (0.30801453564932846-0.6281840090478413j)
(0.3566227649430657-0.5275353392143354j) (-1.7840848650765307e-10-1.0733013294820206e-07j) (0.3566227651214742-0.5275352318842024j)


In [ ]:
(eorb_mp2, eorb_cc), t1, t2 =\
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
print(eorb_cc)

-0.03777829883855721


In [106]:
def ulnoccsd_rand(mcc, uocc_loc, t1, t2):
    prjlo = [uocc_loc[0].T.conj(), uocc_loc[1].T.conj()] # <lno|frag>
    imp_eris = mcc.ao2mo()
    elcorr_cc = ulnoccsd.get_fragment_energy(imp_eris, t1, t2, prjlo)
    # occidxa = mo_occ[0]>1e-10
    # occidxb = mo_occ[1]>1e-10
    # # nmo = mo_occ[0].size, mo_occ[1].size
    # moidxa, moidxb = maskact

    # orbfrzocca = mo_coeff[0][:, ~moidxa &  occidxa]
    # orbactocca = mo_coeff[0][:,  moidxa &  occidxa]
    # orbactvira = mo_coeff[0][:,  moidxa & ~occidxa]
    # orbfrzvira = mo_coeff[0][:, ~moidxa & ~occidxa]
    # nfrzocca, nactocca, nactvira, nfrzvira = [orb.shape[1]
    #                                           for orb in [orbfrzocca,orbactocca,
    #                                                       orbactvira,orbfrzvira]]
    # orbfrzoccb = mo_coeff[1][:, ~moidxb &  occidxb]
    # orbactoccb = mo_coeff[1][:,  moidxb &  occidxb]
    # orbactvirb = mo_coeff[1][:,  moidxb & ~occidxb]
    # orbfrzvirb = mo_coeff[1][:, ~moidxb & ~occidxb]
    # nfrzoccb, nactoccb, nactvirb, nfrzvirb = [orb.shape[1]
    #                                           for orb in [orbfrzoccb,orbactoccb,
    #                                                       orbactvirb,orbfrzvirb]]
    # nlo = [uocc_loc[0].shape[1], uocc_loc[1].shape[1]]
    # prjlo = [uocc_loc[0].T.conj(), uocc_loc[1].T.conj()]
    # if nactocca * nactvira == 0 and nactoccb * nactvirb == 0:
    #     elcorr_pt2 = lib.tag_array(0., spin_comp=np.array((0., 0.)))
    #     elcorr_cc = lib.tag_array(0., spin_comp=np.array((0., 0.)))
    # else:
    #     # solve CCSD impurity problem
    #     imp_eris = mcc.ao2mo()
    #     # MP2 fragment energy
    #     t1, t2 = mcc.init_amps(eris=imp_eris)[1:]
    #     elcorr_pt2 = ulnoccsd.get_fragment_energy(imp_eris, t1, t2, prjlo)
    #     # CCSD fragment energy
    #     t1, t2 = mcc.kernel(eris=imp_eris, t1=t1, t2=t2)[1:]
    #     t1a, t1b = t1
    #     t2aa, t2ab, t2bb = t2
    #     t1a = np.random.rand(*t1a.shape)
    #     t1b = np.random.rand(*t1b.shape)
    #     t2aa = np.random.rand(*t2aa.shape)
    #     t2ab = np.random.rand(*t2ab.shape)
    #     t2bb = np.random.rand(*t2bb.shape)
    #     t1 = (t1a, t1b)
    #     t2 = (t2aa, t2ab, t2bb)
    #     elcorr_cc = ulnoccsd.get_fragment_energy(imp_eris, t1, t2, prjlo)

    return elcorr_cc

In [102]:
uocc_loc[0].shape

(7, 5)

In [101]:
frag1_lno[0].shape

(5, 7)

In [104]:
t1a, t1b = t1
t2aa, t2ab, t2bb = t2
t1a = np.random.rand(*t1a.shape)
t1b = np.random.rand(*t1b.shape)
t2aa = np.random.rand(*t2aa.shape)
t2ab = np.random.rand(*t2ab.shape)
t2bb = np.random.rand(*t2bb.shape)
t1 = (t1a, t1b)
t2 = (t2aa, t2ab, t2bb)

In [115]:
eorbcc1 =\
        ulnoccsd_rand(mcc, (frag1_lno[0].T,frag1_lno[1].T), t1, t2)
print(eorbcc1)
eorbcc2 =\
        ulnoccsd_rand(mcc, (frag2_lno[0].T,frag2_lno[1].T), t1, t2)
print(eorbcc2)
print(eorbcc1+eorbcc2)
eorbcc3 =\
        ulnoccsd_rand(mcc, id, t1, t2)
print(eorbcc3)

-0.03459441332455482


-0.06922378358518724
-0.10381819690974206
-0.10381819690974205


In [93]:
print(f"CCSD per O : {(mycc.e_tot-mf.e_tot)/nc/2}")

CCSD per O : -0.037778302329513735
